In [1]:
import cda2
import json
import re
import pyspark.sql.types as T
import pyspark.sql.functions as F
#import datetime
import time

from datetime import datetime, timedelta
from timeit import default_timer as timer
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

# Connect to Spark

In [2]:
api = cda2.Api(timeout=120)

Set configuration parameters to better optimize queries.

In [3]:
config = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1m",
    "spark.executor.memory": "16g",
    "spark.executor.memoryOverhead": "16g",
    "spark.reducer.maxReqsInFlight": "1",
    "spark.shuffle.io.retryWait": "60s",
    "spark.shuffle.io.maxRetries": "10",
}

# config = {
#     # "spark.sql.adaptive.enabled": "true",
#     # "spark.sql.adaptive.coalescePartitions.enabled": "true",
#     # "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
#     # "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1m",
#     "spark.executor.memory": "16g",
#     "spark.executor.memoryOverhead": "16g",
# }

Start Spark and specify number of cpus to use. 400 is quite high, but we'll be running 1 year at a time and want to have it done in just a few minutes.

In [4]:
api.start_spark(n_executors=400, config=config)
#api.start_spark(n_executors=400)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


24/04/30 11:06:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/04/30 11:06:21 WARN DomainSocketFactory: The short-circuit local reads feature cannot be used because libhadoop cannot be loaded.
24/04/30 11:07:32 WARN DFSClient: Slow waitForAckedSeqno took 65126ms (threshold=30000ms)


In [5]:
api.spark.sparkContext.setLogLevel("ERROR")

In [6]:
year0 = "2023"
year1 = str(int(year0) + 1)

In [7]:
datelist = [
    year0 + "0101",
    year1 + "0101",
]

datelist = [
    year0 + "0101",
    year0 + "0401",
    year0 + "0701",
    year0 + "1001",
    year1 + "0101",
]

datelist = [
    year0 + "0101",
    year0 + "0201",
    year0 + "0301",
    year0 + "0401",
    year0 + "0501",
    year0 + "0601",
    year0 + "0701",
    year0 + "0801",
    year0 + "0901",
    year0 + "1001",
    year0 + "1101",
    year0 + "1201",
    year1 + "0101",
]

In [8]:
script = "retrieve_traffic_v3.ipynb"

In [9]:
i = 0
do_litetracks = True
do_flightplans = True
do_airspace_assignments = True

In [10]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

dates:  {'start_date': '20230101', 'end_date': '20230201'}
start time:  2024-04-30 11:08:06.086483
starting:  {'start_date': '20230101', 'end_date': '20230201'}
   retrieving litetracks:  2024-04-30 11:08:06.210757


Multiple versions found: 3.1.51, 3.1.56, 3.1.64


   retrieving planned_routes:  2024-04-30 11:09:34.993644


Multiple versions found: 3.1.49, 3.1.50
Multiple versions found: 3.1.51, 3.1.52, 3.1.53, 3.1.65


   retrieving flightplans:  2024-04-30 11:09:37.470671


...

24/04/30 11:14:28 ERROR TransportClient: Failed to send RPC RPC 7773249254227027121 to /192.168.164.133:39034: io.netty.channel.StacklessClosedChannelException
io.netty.channel.StacklessClosedChannelException
	at io.netty.channel.AbstractChannel$AbstractUnsafe.write(Object, ChannelPromise)(Unknown Source)
24/04/30 11:14:28 ERROR TransportClient: Failed to send RPC RPC 6488490979220051293 to /192.168.164.215:39408: io.netty.channel.StacklessClosedChannelException
io.netty.channel.StacklessClosedChannelException
	at io.netty.channel.AbstractChannel$AbstractUnsafe.write(Object, ChannelPromise)(Unknown Source)
24/04/30 11:14:28 ERROR TransportClient: Failed to send RPC RPC 8494906096696481906 to /192.168.164.92:56060: io.netty.channel.StacklessClosedChannelException
io.netty.channel.StacklessClosedChannelException
	at io.netty.channel.AbstractChannel$AbstractUnsafe.write(Object, ChannelPromise)(Unknown Source)
24/04/30 11:14:28 ERROR TransportResponseHandler: Still have 1 requests outstand

      df_master size: 887804 2024-04-30 11:16:14.163602
   started saving litetracks:  2024-04-30 11:16:14.785960


   completed saving litetracks:  2024-04-30 12:38:38.401250
   started saving flightplans:  2024-04-30 12:38:38.404020


   completed saving flightplans:  2024-04-30 13:34:06.797742
   started saving airspace_assignments:  2024-04-30 13:34:06.802648


   completed saving airspace_assignments:  2024-04-30 15:24:38.104199
end time:  2024-04-30 15:24:38.105352


In [11]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

dates:  {'start_date': '20230201', 'end_date': '20230301'}
start time:  2024-04-30 15:24:38.113593
starting:  {'start_date': '20230201', 'end_date': '20230301'}
   retrieving litetracks:  2024-04-30 15:24:38.121909


Multiple versions found: 3.1.53, 3.1.64


   retrieving planned_routes:  2024-04-30 15:25:45.645088


Multiple versions found: 3.1.50, 3.1.51
Multiple versions found: 3.1.51, 3.1.52
Multiple versions found: 3.1.51, 3.1.64


   retrieving flightplans:  2024-04-30 15:25:47.437394
   retrieving airspace_assignments:  2024-04-30 15:25:48.212204


...

   joining df_litetracks to df_flightplanseries:  2024-04-30 15:26:17.788316
   joining in df_plannedroutes:  2024-04-30 15:26:17.798539
   joining df_airspaceassignments:  2024-04-30 15:26:17.811892
   creating master dataframe:  2024-04-30 15:26:17.976566


      df_master size: 884889 2024-04-30 15:28:57.938244
   started saving litetracks:  2024-04-30 15:28:58.362522


   completed saving litetracks:  2024-04-30 16:57:09.501230
   started saving flightplans:  2024-04-30 16:57:09.503345


   completed saving flightplans:  2024-04-30 18:13:42.475653
   started saving airspace_assignments:  2024-04-30 18:13:42.482224


   completed saving airspace_assignments:  2024-04-30 19:22:02.731883
end time:  2024-04-30 19:22:02.732626


In [12]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

dates:  {'start_date': '20230301', 'end_date': '20230401'}
start time:  2024-04-30 19:22:02.741900
starting:  {'start_date': '20230301', 'end_date': '20230401'}
   retrieving litetracks:  2024-04-30 19:22:02.749664


Multiple versions found: 3.1.53, 3.1.54, 3.1.55


   retrieving planned_routes:  2024-04-30 19:22:19.981697


Multiple versions found: 3.1.51, 3.1.53
Multiple versions found: 3.1.52, 3.1.53, 3.1.54
Multiple versions found: 3.1.52, 3.1.53, 3.1.54, 3.1.64


   retrieving flightplans:  2024-04-30 19:22:21.259863
   retrieving airspace_assignments:  2024-04-30 19:22:21.874160


...

   joining df_litetracks to df_flightplanseries:  2024-04-30 19:22:51.777001
   joining in df_plannedroutes:  2024-04-30 19:22:51.786869
   joining df_airspaceassignments:  2024-04-30 19:22:51.800100
   creating master dataframe:  2024-04-30 19:22:51.963674


      df_master size: 1022131 2024-04-30 19:27:02.465342
   started saving litetracks:  2024-04-30 19:27:02.875320


   completed saving litetracks:  2024-04-30 20:45:49.621392
   started saving flightplans:  2024-04-30 20:45:49.626984


   completed saving flightplans:  2024-04-30 21:52:26.904727
   started saving airspace_assignments:  2024-04-30 21:52:26.911451


   completed saving airspace_assignments:  2024-04-30 23:00:06.765205
end time:  2024-04-30 23:00:06.765944


In [13]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

dates:  {'start_date': '20230401', 'end_date': '20230501'}
start time:  2024-04-30 23:00:06.782712
starting:  {'start_date': '20230401', 'end_date': '20230501'}
   retrieving litetracks:  2024-04-30 23:00:06.799850


Multiple versions found: 3.1.54, 3.1.55, 3.1.56


   retrieving planned_routes:  2024-04-30 23:02:36.291415


Multiple versions found: 3.1.53, 3.1.54
Multiple versions found: 3.1.54, 3.1.56


   retrieving flightplans:  2024-04-30 23:02:38.791781


...

   joining df_litetracks to df_flightplanseries:  2024-04-30 23:03:12.571959
   joining in df_plannedroutes:  2024-04-30 23:03:12.580288
   joining df_airspaceassignments:  2024-04-30 23:03:12.592107
   creating master dataframe:  2024-04-30 23:03:12.733051


      df_master size: 978155 2024-04-30 23:09:01.925295
   started saving litetracks:  2024-04-30 23:09:02.313184


   completed saving litetracks:  2024-05-01 00:22:49.499796
   started saving flightplans:  2024-05-01 00:22:49.506110


   completed saving flightplans:  2024-05-01 01:35:22.425009
   started saving airspace_assignments:  2024-05-01 01:35:22.431805


   completed saving airspace_assignments:  2024-05-01 02:44:44.999373
end time:  2024-05-01 02:44:45.000090


In [15]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

dates:  {'start_date': '20230501', 'end_date': '20230601'}
start time:  2024-05-01 06:50:05.561057
starting:  {'start_date': '20230501', 'end_date': '20230601'}
   retrieving litetracks:  2024-05-01 06:50:05.574149


Multiple versions found: 3.1.55, 3.1.57, 3.1.58
Multiple versions found: 3.1.54, 3.1.55                                         


   retrieving planned_routes:  2024-05-01 06:50:14.907910


Multiple versions found: 3.1.55, 3.1.56, 3.1.57


   retrieving flightplans:  2024-05-01 06:50:16.266445


...

      df_master size: 1023788 2024-05-01 06:54:07.816619
   started saving litetracks:  2024-05-01 06:54:08.254695


24/05/01 06:55:36 ERROR TransportClient: Failed to send RPC RPC 8313197886582153398 to /192.168.164.206:49820: io.netty.channel.StacklessClosedChannelException
io.netty.channel.StacklessClosedChannelException
	at io.netty.channel.AbstractChannel$AbstractUnsafe.write(Object, ChannelPromise)(Unknown Source)


   completed saving litetracks:  2024-05-01 08:01:00.798517
   started saving flightplans:  2024-05-01 08:01:00.803996


   completed saving flightplans:  2024-05-01 08:36:13.799851
   started saving airspace_assignments:  2024-05-01 08:36:13.802575


   completed saving airspace_assignments:  2024-05-01 09:46:23.905142
end time:  2024-05-01 09:46:23.905890


In [16]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

dates:  {'start_date': '20230601', 'end_date': '20230701'}
start time:  2024-05-01 09:46:23.915845
starting:  {'start_date': '20230601', 'end_date': '20230701'}
   retrieving litetracks:  2024-05-01 09:46:23.922946


Multiple versions found: 3.1.57, 3.1.58, 3.1.59
Multiple versions found: 3.1.55, 3.1.58                                         


   retrieving planned_routes:  2024-05-01 09:46:40.443479


Multiple versions found: 3.1.57, 3.1.58
Multiple versions found: 3.1.57, 3.1.58


   retrieving flightplans:  2024-05-01 09:46:41.488960
   retrieving airspace_assignments:  2024-05-01 09:46:42.083561


...

   joining df_litetracks to df_flightplanseries:  2024-05-01 09:47:09.218780
   joining in df_plannedroutes:  2024-05-01 09:47:09.225967
   joining df_airspaceassignments:  2024-05-01 09:47:09.237301
   creating master dataframe:  2024-05-01 09:47:09.423011


      df_master size: 997132 2024-05-01 09:51:04.625328
   started saving litetracks:  2024-05-01 09:51:04.985227


   completed saving litetracks:  2024-05-01 11:33:50.963654
   started saving flightplans:  2024-05-01 11:33:50.969042


   completed saving flightplans:  2024-05-01 12:49:08.416482
   started saving airspace_assignments:  2024-05-01 12:49:08.419144


   completed saving airspace_assignments:  2024-05-01 14:03:27.933650
end time:  2024-05-01 14:03:27.934235


In [17]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

dates:  {'start_date': '20230701', 'end_date': '20230801'}
start time:  2024-05-01 14:03:27.949880
starting:  {'start_date': '20230701', 'end_date': '20230801'}
   retrieving litetracks:  2024-05-01 14:03:27.958457


Multiple versions found: 3.1.59, 3.1.60
Multiple versions found: 3.1.58, 3.1.59                                         


   retrieving planned_routes:  2024-05-01 14:03:43.805210


Multiple versions found: 3.1.59, 3.1.60
Multiple versions found: 3.1.59, 3.1.60


   retrieving flightplans:  2024-05-01 14:03:45.191023


...

24/05/01 14:08:06 ERROR TransportClient: Failed to send RPC RPC 8053642647924187680 to /192.168.164.213:36666: io.netty.channel.StacklessClosedChannelException
io.netty.channel.StacklessClosedChannelException
	at io.netty.channel.AbstractChannel$AbstractUnsafe.write(Object, ChannelPromise)(Unknown Source)
24/05/01 14:08:06 ERROR TransportClient: Failed to send RPC RPC 7600094381712506403 to /192.168.164.191:34546: io.netty.channel.StacklessClosedChannelException
io.netty.channel.StacklessClosedChannelException
	at io.netty.channel.AbstractChannel$AbstractUnsafe.write(Object, ChannelPromise)(Unknown Source)
24/05/01 14:08:06 ERROR TransportClient: Failed to send RPC RPC 4881852357938163777 to /192.168.164.167:54956: io.netty.channel.StacklessClosedChannelException
io.netty.channel.StacklessClosedChannelException
	at io.netty.channel.AbstractChannel$AbstractUnsafe.write(Object, ChannelPromise)(Unknown Source)
24/05/01 14:08:06 ERROR TransportClient: Failed to send RPC RPC 788613482822576

      df_master size: 1027063 2024-05-01 14:09:17.014229
   started saving litetracks:  2024-05-01 14:09:17.378074


   completed saving litetracks:  2024-05-01 15:29:18.461344
   started saving flightplans:  2024-05-01 15:29:18.463654


   completed saving flightplans:  2024-05-01 16:15:06.785531
   started saving airspace_assignments:  2024-05-01 16:15:06.790129


   completed saving airspace_assignments:  2024-05-01 17:26:30.088845
end time:  2024-05-01 17:26:30.089529


In [18]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

dates:  {'start_date': '20230801', 'end_date': '20230901'}
start time:  2024-05-01 17:26:30.097620
starting:  {'start_date': '20230801', 'end_date': '20230901'}
   retrieving litetracks:  2024-05-01 17:26:30.105889


Multiple versions found: 3.1.60, 3.1.61, 3.1.62


   retrieving planned_routes:  2024-05-01 17:28:21.318266


Multiple versions found: 3.1.59, 3.1.60
Multiple versions found: 3.1.60, 3.1.61


   retrieving flightplans:  2024-05-01 17:28:22.794445


...

   joining df_litetracks to df_flightplanseries:  2024-05-01 17:29:50.474661
   joining in df_plannedroutes:  2024-05-01 17:29:50.484739
   joining df_airspaceassignments:  2024-05-01 17:29:50.503943
   creating master dataframe:  2024-05-01 17:29:50.716629


      df_master size: 1044500 2024-05-01 17:36:55.286821
   started saving litetracks:  2024-05-01 17:36:55.686298


   completed saving litetracks:  2024-05-01 18:33:58.884374
   started saving flightplans:  2024-05-01 18:33:58.887316


   completed saving flightplans:  2024-05-01 19:58:23.424887
   started saving airspace_assignments:  2024-05-01 19:58:23.429426


   completed saving airspace_assignments:  2024-05-01 21:07:31.374960
end time:  2024-05-01 21:07:31.375366


In [20]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

dates:  {'start_date': '20230901', 'end_date': '20231001'}
start time:  2024-05-01 21:30:21.489393
starting:  {'start_date': '20230901', 'end_date': '20231001'}
   retrieving litetracks:  2024-05-01 21:30:21.499603


Multiple versions found: 3.1.61, 3.1.62


   retrieving planned_routes:  2024-05-01 21:30:22.749873


Multiple versions found: 3.1.60, 3.1.61, 3.1.62
Multiple versions found: 3.1.60, 3.1.61, 3.1.62                                 


   retrieving flightplans:  2024-05-01 21:30:25.386149
   retrieving airspace_assignments:  2024-05-01 21:30:26.298102


...

   joining df_litetracks to df_flightplanseries:  2024-05-01 21:30:29.769906
   joining in df_plannedroutes:  2024-05-01 21:30:29.785554
   joining df_airspaceassignments:  2024-05-01 21:30:29.799153
   creating master dataframe:  2024-05-01 21:30:29.978762


      df_master size: 989740 2024-05-01 21:34:01.374559
   started saving litetracks:  2024-05-01 21:34:01.820102


   completed saving litetracks:  2024-05-01 22:51:24.613414
   started saving flightplans:  2024-05-01 22:51:24.618858


   completed saving flightplans:  2024-05-01 23:30:32.677080
   started saving airspace_assignments:  2024-05-01 23:30:32.679403


   completed saving airspace_assignments:  2024-05-02 00:38:11.542064
end time:  2024-05-02 00:38:11.542564


In [21]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

dates:  {'start_date': '20231001', 'end_date': '20231101'}
start time:  2024-05-02 00:38:11.552862
starting:  {'start_date': '20231001', 'end_date': '20231101'}
   retrieving litetracks:  2024-05-02 00:38:11.560818


Multiple versions found: 3.1.62, 3.1.63


   retrieving planned_routes:  2024-05-02 00:38:21.437186


Multiple versions found: 3.1.60, 3.1.62
Multiple versions found: 3.1.62, 3.1.63                                         


   retrieving flightplans:  2024-05-02 00:38:24.784636


...

   joining df_litetracks to df_flightplanseries:  2024-05-02 00:38:57.121755
   joining in df_plannedroutes:  2024-05-02 00:38:57.131762
   joining df_airspaceassignments:  2024-05-02 00:38:57.144855
   creating master dataframe:  2024-05-02 00:38:57.323337


      df_master size: 1039098 2024-05-02 00:42:07.951923
   started saving litetracks:  2024-05-02 00:42:08.324786


   completed saving litetracks:  2024-05-02 01:52:48.657633
   started saving flightplans:  2024-05-02 01:52:48.661391


   completed saving flightplans:  2024-05-02 02:56:48.798756
   started saving airspace_assignments:  2024-05-02 02:56:48.804669


   completed saving airspace_assignments:  2024-05-02 04:02:37.422432
end time:  2024-05-02 04:02:37.422818


In [22]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

dates:  {'start_date': '20231101', 'end_date': '20231201'}
start time:  2024-05-02 04:02:37.429762
starting:  {'start_date': '20231101', 'end_date': '20231201'}
   retrieving litetracks:  2024-05-02 04:02:37.448797


Multiple versions found: 3.1.62, 3.1.63


   retrieving planned_routes:  2024-05-02 04:02:47.252025


Multiple versions found: 3.1.62, 3.1.63
Multiple versions found: 3.1.62, 3.1.63


   retrieving flightplans:  2024-05-02 04:02:48.605763


...

   joining df_litetracks to df_flightplanseries:  2024-05-02 04:03:12.879645
   joining in df_plannedroutes:  2024-05-02 04:03:12.888234
   joining df_airspaceassignments:  2024-05-02 04:03:12.900341
   creating master dataframe:  2024-05-02 04:03:13.061123


      df_master size: 995015 2024-05-02 04:06:22.202933
   started saving litetracks:  2024-05-02 04:06:22.559790


   completed saving litetracks:  2024-05-02 04:50:26.619286
   started saving flightplans:  2024-05-02 04:50:26.625266


   completed saving flightplans:  2024-05-02 05:53:46.941512
   started saving airspace_assignments:  2024-05-02 05:53:46.947935


   completed saving airspace_assignments:  2024-05-02 06:55:07.416188
end time:  2024-05-02 06:55:07.416878


In [23]:
if i < len(datelist) - 1:
    dates = {"start_date": datelist[i], "end_date": datelist[i+1]}
    print("dates: ", dates)
    print("start time: ", datetime.now())
    %run $script
    i += 1
    print("end time: ", datetime.now())

dates:  {'start_date': '20231201', 'end_date': '20240101'}
start time:  2024-05-02 06:55:07.436931
starting:  {'start_date': '20231201', 'end_date': '20240101'}
   retrieving litetracks:  2024-05-02 06:55:07.444536


Multiple versions found: 3.1.63, 3.1.64, 3.1.65


   retrieving planned_routes:  2024-05-02 06:55:16.780936


Multiple versions found: 3.1.63, 3.1.64
Multiple versions found: 3.1.63, 3.1.64, 3.1.65


   retrieving flightplans:  2024-05-02 06:55:18.374256


...

      df_master size: 1020076 2024-05-02 06:57:47.432157
   started saving litetracks:  2024-05-02 06:57:47.801574


24/05/02 07:00:11 ERROR ContextCleaner: Error cleaning broadcast 904
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:301)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.storage.BlockManagerMaster.removeBroadcast(BlockManagerMaster.scala:195)
	at org.apache.spark.broadcast.TorrentBroadcast$.unpersist(TorrentBroadcast.scala:351)
	at org.apache.spark.broadcast.TorrentBroadcastFactory.unbroadcast(TorrentBroadcastFactory.scala:45)
	at org.apache.spark.broadcast.BroadcastManager.unbroadcast(BroadcastManager.scala:79)
	at org.apache.spark.ContextCleaner.doCleanupBroadcast(ContextCleaner.scala:256)
	at org.apache.spark.ContextCleaner.$anonfun$keepCleaning$3(ContextCleaner.scala:204)
	at org.apache.spark.ContextCleaner.$anonfun$keepCleaning$3$adapted(ContextCleaner.scala:195)
	at scala.Option.foreach(Option.scala:407)
	at org.apache.spark.ContextCleaner.$anonfun$kee

   completed saving litetracks:  2024-05-02 08:03:31.603349
   started saving flightplans:  2024-05-02 08:03:31.609209


   completed saving flightplans:  2024-05-02 09:03:25.862627
   started saving airspace_assignments:  2024-05-02 09:03:25.864877


   completed saving airspace_assignments:  2024-05-02 10:07:19.123077
end time:  2024-05-02 10:07:19.123748
